# A: aligned course direction vs time (shuffled and synthetic trajectories)

Fills layer `unifying` of the figure SVG. Run A → C → D (any order; each fills its own layer).

In [1]:
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Helvetica'] + matplotlib.rcParams['font.sans-serif']
matplotlib.rcParams['font.size'] = 6
matplotlib.rcParams['text.usetex'] = False
matplotlib.rcParams["ps.usedistiller"] = 'xpdf'
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.weight'] = 'normal'
matplotlib.rcParams["mathtext.fontset"] = 'cm'

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

import figurefirst as fifi

from align_course_direction_analysis import unifying_algo_analysis as uaa
from align_course_direction_analysis import unifying_algo_plots as uap
from splitflow.set_zorder_functions import *

from scenarios import FIGURE_NAME, SCENARIOS, load_scenario, save_layer

/Users/florisvb/src/SplitFlowOptoTracking/.venv/lib/python3.11/site-packages/figurefirst/mpl_fig_to_figurefirst_svg.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Using device: cpu


In [3]:
def mean_angle(angle):
    mean = np.arctan2( np.nanmean(np.sin(angle)), np.nanmean(np.cos(angle)) )
    return mean

def angle_distance(angle1, angle2):
    """Minimum distance between two angles (radians), wrapped to [-pi, pi]."""
    diff = angle1 - angle2
    distance = np.arctan2(np.sin(diff), np.cos(diff))
    return distance

def parametric_model(unifying_algo_data, ix_master):
    """
    Model course vs. aligned time built from the median fit parameters (the pink line), instead of the circular
    mean of all aligned affine fits. Uses each trajectory's best fit (lowest rmse_affine), normalized to a positive
    slope as in uaa.get_aligned_course_array (mirror: flip the sign of the off-diagonal AM terms and of tx).
    Model: phi = median|slope| * aligned_ix (phase 0 at aligned t=0, a 0-crossing of the linear fit),
           course = atan2(w0, w1), w = median(AM) @ [sin(phi), cos(phi)] + median([tx, ty])   ((sin, cos) convention)
    """
    best = (unifying_algo_data.sort_values('rmse_affine', kind='stable')
            .groupby('obj_id_unique_event', sort=False).head(1))
    flip = np.diag([-1.0, 1.0])
    slopes, AMs, ts = [], [], []
    for _, row in best.iterrows():
        AM = np.array([[row.AM_0_0, row.AM_0_1], [row.AM_1_0, row.AM_1_1]])
        t = np.nan_to_num(np.array([row.tx, row.ty], dtype=float))  # fits without translation store None
        if row.slope < 0:
            AM, t = flip @ AM @ flip, flip @ t
        slopes.append(abs(row.slope)); AMs.append(AM); ts.append(t)
    phi = np.median(slopes) * np.asarray(ix_master, dtype=float)
    w = np.median(AMs, axis=0) @ np.vstack((np.sin(phi), np.cos(phi))) + np.median(ts, axis=0)[:, None]
    return np.arctan2(w[0], w[1])


In [4]:
def clean_labels(ax, show_labels, spines=['left', 'bottom']):
    ax.set_rasterization_zorder(0)

    ax.set_yticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
    ax.set_ylim(-np.pi, np.pi)
    ax.set_xlim(-2, 2)
    ax.set_xticks([-200, -100, 0, 100, 200])

    if show_labels:
        ax.set_xticklabels(['-2', '-1', '0', '1', '2'])
    else:
        ax.set_xticklabels([])

    if show_labels:
        ax.set_yticklabels(['$-\\pi$', '','$0$','','$\\pi$',])
    else:
        ax.set_yticklabels([])

    fifi.mpl_functions.adjust_spines(ax, ['left', 'bottom'])

    if show_labels:
        ax.set_ylabel('Course direction', labelpad=-2)
        ax.set_xlabel('Aligned time (s)', labelpad=1)
    else:
        ax.set_ylabel('')
        ax.set_xlabel('')

    ax.tick_params(axis='y', pad=2)
    ax.tick_params(axis='x', pad=2)

    fifi.mpl_functions.adjust_spines(ax, ['left', 'bottom'],
                                     tick_length=2.5,
                                     spine_locations={'left': 5, 'bottom': 5},
                                     linewidth=0.5)
    fifi.mpl_functions.set_fontsize(ax, 6)

In [5]:
COURSE_MARKER_SIZE = 2
COURSE_ALPHA_MULTIPLIER = 3

# The pink model line is drawn from this aligned time on (ix units: 100 = 1 s), as in Figure 5
MODEL_START_IX = -100  # -1 s

fifi_figure_label = 'unifying'

In [6]:
layout = fifi.svg_to_axes.FigureLayout(FIGURE_NAME, autogenlayers=True, make_mplfigures=True, hide_layers=[], dpi=600)
plt.close('all')

In [7]:
for key in SCENARIOS:
    unifying_algo_data, df = load_scenario(key)
    ax = layout.axes[(fifi_figure_label, key)]

    # build aligned arrays
    results = uaa.get_aligned_course_array(unifying_algo_data, df, return_linear_and_affine_fits=True)
    ix_master, aligned_time_since_flash_arr, aligned_course_arr, aligned_roi_arr, aligned_linear_fits, aligned_affine_fits = results

    # Model line (pink): built from the median fit parameters, shown from aligned t = -1 s on
    model_line = np.atleast_2d(parametric_model(unifying_algo_data, ix_master))
    model_line[:, np.asarray(ix_master) < MODEL_START_IX] = np.nan

    uap.plot_aligned_course_from_array(ix_master,
                                       aligned_time_since_flash_arr,
                                       aligned_course_arr,
                                       aligned_roi_arr,
                                       aligned_flash_arr=None,
                                       aligned_linear_fits=None, # if None, skips plot
                                       aligned_affine_fits=model_line, # if None, skips plot
                                       xlim_start=-300,
                                       xlim_end=300,
                                       ax=ax,
                                       clean_spines=True,
                                       course_marker_size=COURSE_MARKER_SIZE,
                                       course_alpha_multiplier=COURSE_ALPHA_MULTIPLIER,
                                       )

    clean_labels(ax, show_labels=(key == SCENARIOS[0]))

    N_trajecs = len(unifying_algo_data.obj_id_unique_event.unique())
    layout.svgitems['text_n_' + key + '_unifying'].text = 'n=' + str(N_trajecs)
    print(key, 'n =', N_trajecs)

shuffled_laminar n = 200


shuffled_stillair n = 200


levy_flight n = 200


random_turns n = 200


brownian n = 200


run_and_tumble n = 200


In [8]:
layout.apply_svg_attrs()
save_layer(layout, fifi_figure_label)